In [9]:
import re
import pandas as pd
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

In [10]:
data = pd.read_csv("results---From---2023-10-31--22-08-07---To---2024-04-07--16-02-28.csv")
data = data[['message']].dropna()
data = data.drop_duplicates()
data

,message
0,#Renta x DÍAS de Apto en el Vedado
1,No tienes permisos para ejecutar este comando ...
2,/revisarbrplus@ReputacionPlusBot
8,Casa en venta en la zona sur cerca de las fábr...
9,Busco renta por tiempo indefinido para una par...
...,...
4988,Busco alquiler en el vedado límite 150 verde s...
4992,"Busco alquiler por tiempo indefinido, 58316712"
4993,Busco alquiler en la lisa o lo más cerca posible
4994,Busco alquiler en la Lisa


In [11]:
def delete_emojis(text):
    patron_emojis = re.compile(pattern="["
                                      u"\U0001F600-\U0001F64F"  
                                      u"\U0001F300-\U0001F5FF"  
                                      u"\U0001F680-\U0001F6FF"  
                                      u"\U0001F700-\U0001F77F"  
                                      u"\U0001F780-\U0001F7FF"  
                                      u"\U0001F800-\U0001F8FF"  
                                      u"\U0001F900-\U0001F9FF"  
                                      u"\U0001FA00-\U0001FAFF" 
                                      u"\U00002702-\U000027B0"  
                                      u"\U00002702-\U000027B0"
                                      u"\U000024C2-\U0001F251"
                                      "]+", flags=re.UNICODE)
    return patron_emojis.sub(r'', text)

def delete_commands(texto):
    cleaned_text = re.sub(r'/\S+', '', texto)
    cleaned_text = re.sub(r'@\S+', '', cleaned_text)
    cleaned_text = re.sub(r'#', '', cleaned_text)
    cleaned_text = re.sub(r'http[s]?://\S+', '', cleaned_text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

def normalize_text(texto):
    return texto.lower()


In [12]:
data['message'] = data['message'].apply(delete_emojis)
data['message'] = data['message'].apply(delete_commands) 
data['message'] = data['message'].apply(normalize_text)

In [16]:
output_dir = "./fine_tuned_model"
model = AutoModelForTokenClassification.from_pretrained(output_dir)
tokenizer = AutoTokenizer.from_pretrained(output_dir)
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

def extract_locations(text):
    if pd.isna(text):
        return []

    entities = ner_pipeline(text)
    print(f"Entities: {entities}") 
    locations = []
    current_entity = []

    for entity in entities:
        if entity['entity_group'] in ['LABEL_1', 'LABEL_2']:
            if not current_entity or entity['entity_group'] == 'LABEL_2':
                current_entity.append(entity['word'])
            else:
                locations.append(" ".join(current_entity))
                current_entity = [entity['word']]
        else:
            if current_entity:
                locations.append(" ".join(current_entity))
                current_entity = []
    if current_entity:
        locations.append(" ".join(current_entity))

    return locations


data["locations"] = data["message"].apply(extract_locations)
print(data)

Device set to use cpu


Entities: [{'entity_group': 'LABEL_0', 'score': np.float32(0.9883216), 'word': 'renta x días de apto en', 'start': 0, 'end': 23}, {'entity_group': 'LABEL_1', 'score': np.float32(0.9766259), 'word': 'el vedado', 'start': 24, 'end': 33}]
Entities: [{'entity_group': 'LABEL_0', 'score': np.float32(0.87291497), 'word': 'no tienes permisos para ejecutar este comando aquí.', 'start': 0, 'end': 51}]
Entities: []
Entities: [{'entity_group': 'LABEL_0', 'score': np.float32(0.9923864), 'word': 'casa en venta en la zona sur cerca de las fábricas de cerveza y galleta, hospital pediátrico, especialidades, policlinico sur, hogar de ancianos con agua permanente, poca afectación eléctrica cuenta con portal, sala, cocina, 3 cuartos grande, baño sanitario, pasillo lateral, patio con árboles frutales, corral de puercos, baño de servicio y se deja con refrigerador, tv con cajita su multimueble, muebles, camas, escaparate y libreta de comida precio : 2800 usd me ajusto con dinero en mano, puede pagar por zel

In [28]:
example = "Habitación disponible cerca del fajardo"
entities = ner_pipeline(example)
print(entities)
print(extract_locations(example))

[{'entity_group': 'LABEL_0', 'score': np.float32(0.9609927), 'word': 'habitación disponible cerca del', 'start': 0, 'end': 31}, {'entity_group': 'LABEL_1', 'score': np.float32(0.695378), 'word': 'fajardo', 'start': 32, 'end': 39}]
Entities: [{'entity_group': 'LABEL_0', 'score': np.float32(0.9609927), 'word': 'habitación disponible cerca del', 'start': 0, 'end': 31}, {'entity_group': 'LABEL_1', 'score': np.float32(0.695378), 'word': 'fajardo', 'start': 32, 'end': 39}]
['fajardo']
